# Linalool exposure in *Fusarium oxysporum*: a published-table audit

**Question:** Which metabolic and signaling responses to terpene exposure are reproducible, and which depend on cellular context or a specific target?

This notebook examines one independent fungal linalool cohort (PRJNA913446) from the [primary paper](https://pmc.ncbi.nlm.nih.gov/articles/PMC9820380/) and its [publisher-hosted supplement](https://mdpi-res.com/d_attachment/ijms/ijms-24-00458/article_deploy/ijms-24-00458-s001.zip). It audits published differential-expression and enrichment tables; it does not re-fit raw RNA-seq counts. The 3-versus-3 biological-replicate design and linalool exposure are reported by the authors. A pathway appearing in both transcript and protein tables is within-study cross-omics support, not independent replication or proof of a direct molecular target.

In [1]:
from pathlib import Path
from io import BytesIO
from zipfile import ZipFile
import hashlib, json
import pandas as pd

ROOT = Path.cwd()
PROJECT = ROOT / 'research' / 'fusarium-linalool-supplement'
SOURCE = PROJECT / 'data' / 'ijms-24-00458-s001.zip'
RESULTS = PROJECT / 'results'
RESULTS.mkdir(parents=True, exist_ok=True)
ZIP_SHA256 = '64eb1c7cf6aeea6e54fab89ad9133914db6a0afdd28f1696242fa8fb5139b653'
WORKBOOK_SHA256 = '7b8692619334467570868fe8456c0dd1fcf21ea0d6bef5c0f854754ff86ed6fb'
if hashlib.sha256(SOURCE.read_bytes()).hexdigest() != ZIP_SHA256:
    raise ValueError('Supplement ZIP checksum changed')
with ZipFile(SOURCE) as archive:
    workbook = archive.read('Supplementary tables.xlsx')
if hashlib.sha256(workbook).hexdigest() != WORKBOOK_SHA256:
    raise ValueError('Supplement workbook checksum changed')

def sheet(name, id_column):
    frame = pd.read_excel(BytesIO(workbook), sheet_name=name, header=1)
    return frame.dropna(subset=[id_column]).copy()

all_genes = sheet('Table S3', 'Gene_id')
degs = sheet('Table S4', 'gene_id')
rna_pathways = sheet('Table S6', 'KEGGID')
proteins = sheet('Table S8', 'Protein')
protein_pathways = sheet('Table S10', 'MapID')
print('Supplement SHA-256:', ZIP_SHA256)
print('Workbook SHA-256:', WORKBOOK_SHA256)
print('Gene rows:', len(all_genes), 'published DEG rows:', len(degs))

Supplement SHA-256: 64eb1c7cf6aeea6e54fab89ad9133914db6a0afdd28f1696242fa8fb5139b653
Workbook SHA-256: 7b8692619334467570868fe8456c0dd1fcf21ea0d6bef5c0f854754ff86ed6fb
Gene rows: 13974 published DEG rows: 1568


## Audit the published gene calls

Table S4 is the authors’ differential-expression output, with per-sample processed abundance values, log2 fold changes and adjusted p-values. We check row uniqueness and whether its entries meet the reported cutoff. These are **source-reported DESeq2 results**; the supplement does not contain the raw count matrix or enough model metadata to independently re-estimate uncertainty.

In [2]:
for field in ['Log2FC(linalool/CK)', 'Padj']:
    degs[field] = pd.to_numeric(degs[field], errors='coerce')
if degs['gene_id'].duplicated().any() or not set(degs['gene_id']).issubset(set(all_genes['Gene_id'])):
    raise ValueError('Supplement gene IDs are duplicated or absent from Table S3')
passes_cutoff = degs['Padj'].lt(0.05) & degs['Log2FC(linalool/CK)'].abs().ge(1)
directions = degs['Regulate'].astype(str).str.lower().value_counts().to_dict()
summary = {
    'source_archive_sha256': ZIP_SHA256,
    'source_workbook_sha256': WORKBOOK_SHA256,
    'all_gene_rows': int(len(all_genes)),
    'published_deg_rows': int(len(degs)),
    'published_deg_unique_gene_ids': int(degs['gene_id'].nunique()),
    'published_deg_meeting_adjusted_p_and_effect_cutoff': int(passes_cutoff.sum()),
    'reported_up': int(directions.get('up', 0)),
    'reported_down': int(directions.get('down', 0)),
    'interpretation': 'Audit of author-supplied tables, not a new raw-count differential-expression fit.'
}
print(json.dumps(summary, indent=2))

{
  "source_archive_sha256": "64eb1c7cf6aeea6e54fab89ad9133914db6a0afdd28f1696242fa8fb5139b653",
  "source_workbook_sha256": "7b8692619334467570868fe8456c0dd1fcf21ea0d6bef5c0f854754ff86ed6fb",
  "all_gene_rows": 13974,
  "published_deg_rows": 1568,
  "published_deg_unique_gene_ids": 1568,
  "published_deg_meeting_adjusted_p_and_effect_cutoff": 1568,
  "reported_up": 979,
  "reported_down": 589,
  "interpretation": "Audit of author-supplied tables, not a new raw-count differential-expression fit."
}


## Compare transcript and protein pathway tables

We use the adjusted p-values already present in Tables S6 and S10 (threshold 0.05), matching pathways by the five-digit KEGG identifier suffix. Different prefixes (`fox` and `map`) refer to organism-specific and pathway-map identifiers. Matching a pathway says both published lists enriched that term; it does not imply that the same molecules or directions drove both results.

In [3]:
rna_pathways['padj'] = pd.to_numeric(rna_pathways['padj'], errors='coerce')
protein_pathways['AdjustedPv'] = pd.to_numeric(protein_pathways['AdjustedPv'], errors='coerce')
rna_sig = rna_pathways.loc[rna_pathways['padj'].lt(0.05)].copy()
protein_sig = protein_pathways.loc[protein_pathways['AdjustedPv'].lt(0.05)].copy()
rna_sig['kegg_core'] = rna_sig['KEGGID'].astype(str).str.extract(r'(\d{5})$')[0]
protein_sig['kegg_core'] = protein_sig['MapID'].astype(str).str.extract(r'(\d{5})$')[0]
pathways = rna_sig[['kegg_core', 'KEGGID', 'Description', 'padj', 'geneID']].merge(
    protein_sig[['kegg_core', 'MapID', 'MapTitle', 'AdjustedPv', 'ProtIDs']],
    on='kegg_core', how='outer', validate='one_to_one'
).sort_values('kegg_core')
pathways['significant_in_both'] = pathways['KEGGID'].notna() & pathways['MapID'].notna()
pathways.rename(columns={'padj': 'rna_adjusted_p', 'AdjustedPv': 'protein_adjusted_p',
                         'Description': 'rna_pathway_name', 'MapTitle': 'protein_pathway_name',
                         'geneID': 'rna_gene_ids', 'ProtIDs': 'protein_ids'}, inplace=True)
pathways.to_csv(RESULTS / 'pathway_crossomics.tsv', sep='\t', index=False)
summary['significant_rna_kegg_terms'] = int(len(rna_sig))
summary['significant_protein_kegg_terms'] = int(len(protein_sig))
summary['significant_in_both'] = int(pathways['significant_in_both'].sum())
print(pathways[['kegg_core', 'rna_pathway_name', 'protein_pathway_name', 'rna_adjusted_p', 'protein_adjusted_p', 'significant_in_both']].to_string(index=False))

kegg_core                            rna_pathway_name                        protein_pathway_name  rna_adjusted_p  protein_adjusted_p  significant_in_both
    00100                        Steroid biosynthesis                                         NaN    6.588855e-08                 NaN                False
    00520 Amino sugar and nucleotide sugar metabolism Amino sugar and nucleotide sugar metabolism    1.406906e-04            0.005405                 True
    02010                            ABC transporters                                         NaN    8.505814e-05                 NaN                False


/var/folders/sg/7k1f9lns75sdtd39z4tnhvjr0000gn/T/ipykernel_73672/4252147021.py:12: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  pathways.rename(columns={'padj': 'rna_adjusted_p', 'AdjustedPv': 'protein_adjusted_p',


## Check molecule overlap in the shared pathway

For every KEGG term significant in both tables, map protein identifiers such as `FOXG_03194P0` to their gene stem (`FOXG_03194`). Report exact member overlap and the transcript/protein directions for those shared IDs. This prevents a pathway-level match from being presented as broad gene-to-protein concordance.

In [4]:
rows = []
for pathway in pathways.loc[pathways['significant_in_both']].itertuples(index=False):
    genes = set(str(pathway.rna_gene_ids).split('/'))
    protein_ids = {item.removesuffix('P0'): item for item in str(pathway.protein_ids).split()}
    for gene_id in sorted(genes & set(protein_ids)):
        gene_row = degs.loc[degs['gene_id'].eq(gene_id)].iloc[0]
        protein_row = proteins.loc[proteins['Protein'].eq(protein_ids[gene_id])].iloc[0]
        rows.append({
            'kegg_core': pathway.kegg_core,
            'pathway': pathway.rna_pathway_name,
            'gene_id': gene_id,
            'protein_id': protein_ids[gene_id],
            'rna_log2fc': float(gene_row['Log2FC(linalool/CK)']),
            'rna_adjusted_p': float(gene_row['Padj']),
            'rna_direction': str(gene_row['Regulate']).lower(),
            'protein_log2fc': float(protein_row['Linalool.vs.CK log2FC']),
            'protein_p': float(protein_row['Linalool.vs.CK Pvalue']),
            'protein_direction': str(protein_row['Linalool.vs.CK UP.DOWN']).lower(),
            'direction_agrees': str(gene_row['Regulate']).lower() == str(protein_row['Linalool.vs.CK UP.DOWN']).lower()
        })
overlap = pd.DataFrame(rows)
overlap.to_csv(RESULTS / 'gene_protein_overlap.tsv', sep='\t', index=False)
summary['same_gene_protein_members_in_shared_pathways'] = int(len(overlap))
(RESULTS / 'analysis_summary.json').write_text(json.dumps(summary, indent=2) + '\n')
print(overlap.to_string(index=False))
print('Saved', RESULTS.relative_to(ROOT))

kegg_core                                     pathway    gene_id   protein_id  rna_log2fc  rna_adjusted_p rna_direction  protein_log2fc  protein_p protein_direction  direction_agrees
    00520 Amino sugar and nucleotide sugar metabolism FOXG_03194 FOXG_03194P0   -1.572448        0.012817          down       -1.039277   0.007579              down              True
Saved research/fusarium-linalool-supplement/results


## Interpretation and next experiment

The supplement supports a **within-study** metabolic response to linalool in this fungal context. One KEGG term is significant in both published omics enrichment tables; its member lists share one exact gene/protein identifier, with concordant down direction. This is not cross-study reproduction: the omics came from the same exposure study, and the published DEG/protein tables were not refitted here. We should reprocess PRJNA913446 raw reads with a pinned reference and count model, then seek another time-, dose-, and organism-compatible linalool cohort before generalizing. A direct molecular target requires an additional chemical-genetic interaction or binding experiment.